# B05a: Neural Networks Theory - COMPSCI 714

**Course:** COMPSCI 714 - AI Architecture and Design  
**Lectures:** 2 & 3  

This notebook covers COMPSCI 714 Lecture 2 & 3 topics with Python implementations.

## Topics Covered
- Artificial Neuron
- Perceptron
- Gradient Descent
- Backpropagation

See full guides:
- [Lecture 2 Guide](../documentation/courses/COMPSCI_714_LECTURE_2.md)
- [Lecture 3 Guide](../documentation/courses/COMPSCI_714_LECTURE_3.md)

### Interactive Apps
- [Neural Network Explorer](https://nexageapps.github.io/AI/neural-network/) — Build and visualise neural networks, explore activation functions, and try a single-neuron playground

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print('Notebook ready!')

Notebook ready!


## Part 1: Artificial Neuron

Components: inputs (x), weights (w), bias (b), pre-activation (z), activation g(z), output (a)

In [2]:
# Artificial Neuron Implementation
x = np.array([1.0, 2.0, 3.0])  # inputs
w = np.array([0.5, -0.3, 0.8])  # weights
b = 0.1  # bias

# Pre-activation: z = w·x + b
z = np.dot(w, x) + b
print(f'Pre-activation z: {z:.4f}')

# Activation: a = σ(z)
a = 1 / (1 + np.exp(-z))
print(f'Output a: {a:.4f}')

Pre-activation z: 2.4000
Output a: 0.9168


---

## Exercises

Test your understanding with these hands-on exercises. Try to solve them before looking at the hints.


### Exercise 1: Backpropagation by Hand

For a simple 2-layer network with one hidden unit, compute the forward pass and backward pass manually:
- Input: x = 0.5
- Weights: w1 = 0.3, w2 = 0.7
- Biases: b1 = 0.1, b2 = 0.2
- Activation: sigmoid
- Target: y = 1.0
- Loss: MSE

Compute all gradients (∂L/∂w2, ∂L/∂b2, ∂L/∂w1, ∂L/∂b1) step by step.



In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

x, w1, b1, w2, b2, y_true = 0.5, 0.3, 0.1, 0.7, 0.2, 1.0

# Forward pass
z1 = w1 * x + b1
a1 = sigmoid(z1)

z2 = w2 * a1 + b2
y_pred = sigmoid(z2)

loss = (y_pred - y_true) ** 2

print("FORWARD PASS")
print(f"z1: {z1}")
print(f"a1: {a1}")
print(f"z2: {z2}")
print(f"y_pred: {y_pred}")
print(f"loss: {loss}")

# -----------------------------------
# BACKWARD PASS
# -----------------------------------

# dLoss/dy_pred
dloss_dypred = 2 * (y_pred - y_true)

# dy_pred/dz2
dypred_dz2 = sigmoid_derivative(z2)

# dLoss/dz2
dloss_dz2 = dloss_dypred * dypred_dz2

# Gradients for w2 and b2
dloss_dw2 = dloss_dz2 * a1
dloss_db2 = dloss_dz2

# dz2/da1 = w2
dloss_da1 = dloss_dz2 * w2

# da1/dz1
da1_dz1 = sigmoid_derivative(z1)

# dLoss/dz1
dloss_dz1 = dloss_da1 * da1_dz1

# Gradients for w1 and b1
dloss_dw1 = dloss_dz1 * x
dloss_db1 = dloss_dz1

print("\nBACKWARD PASS")
print(f"dLoss/dw2: {dloss_dw2}")
print(f"dLoss/db2: {dloss_db2}")

print(f"dLoss/dw1: {dloss_dw1}")
print(f"dLoss/db1: {dloss_db1}")

FORWARD PASS
z1: 0.25
a1: 0.5621765008857981
z2: 0.5935235506200587
y_pred: 0.6441732027931297
loss: 0.12661270961049917

BACKWARD PASS
dLoss/dw2: -0.09170280948863031
dLoss/db2: -0.16312102932822348
dLoss/dw1: -0.014052375725120306
dLoss/db1: -0.028104751450240613


### Exercise 2: Vanishing Gradient Demonstration

Create a deep network (10+ layers) using sigmoid activations and observe the vanishing gradient problem. Print the mean absolute gradient for each layer after one training step. Then repeat with ReLU and compare.



In [2]:
import tensorflow as tf
import numpy as np

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Dummy data
X = np.random.randn(64, 20).astype(np.float32)
y = np.random.randn(64, 1).astype(np.float32)

# ---------------------------------------------------
# Function to create deep model
# ---------------------------------------------------
def create_model(activation):

    model = tf.keras.Sequential()

    # 12 hidden layers
    for _ in range(12):
        model.add(
            tf.keras.layers.Dense(
                64,
                activation=activation
            )
        )

    model.add(tf.keras.layers.Dense(1))

    return model

# ---------------------------------------------------
# Function to inspect gradients
# ---------------------------------------------------
def inspect_gradients(activation_name):

    print(f"\n===== {activation_name.upper()} NETWORK =====")

    model = create_model(activation_name)

    optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)

    with tf.GradientTape() as tape:

        predictions = model(X)

        loss = tf.reduce_mean(
            tf.square(predictions - y)
        )
    
    breakpoint()
    gradients = tape.gradient(loss, model.trainable_variables)

    layer_num = 1

    for grad in gradients:

        if grad is not None:

            mean_grad = tf.reduce_mean(tf.abs(grad))

            print(
                f"Layer {layer_num:02d} "
                f"Mean |Gradient|: "
                f"{mean_grad.numpy():.10f}"
            )

            layer_num += 1

# ---------------------------------------------------
# Run experiments
# ---------------------------------------------------
inspect_gradients("sigmoid")

inspect_gradients("relu")


===== SIGMOID NETWORK =====


Layer 01 Mean |Gradient|: 0.0000000005
Layer 02 Mean |Gradient|: 0.0000000043
Layer 03 Mean |Gradient|: 0.0000000093
Layer 04 Mean |Gradient|: 0.0000000184
Layer 05 Mean |Gradient|: 0.0000000337
Layer 06 Mean |Gradient|: 0.0000000670
Layer 07 Mean |Gradient|: 0.0000001350
Layer 08 Mean |Gradient|: 0.0000002624
Layer 09 Mean |Gradient|: 0.0000005941
Layer 10 Mean |Gradient|: 0.0000012369
Layer 11 Mean |Gradient|: 0.0000025829
Layer 12 Mean |Gradient|: 0.0000049677
Layer 13 Mean |Gradient|: 0.0000112154
Layer 14 Mean |Gradient|: 0.0000221829
Layer 15 Mean |Gradient|: 0.0000503117
Layer 16 Mean |Gradient|: 0.0001032246
Layer 17 Mean |Gradient|: 0.0002101174
Layer 18 Mean |Gradient|: 0.0004225301
Layer 19 Mean |Gradient|: 0.0012606994
Layer 20 Mean |Gradient|: 0.0022577478
Layer 21 Mean |Gradient|: 0.0054306583
Layer 22 Mean |Gradient|: 0.0109888949
Layer 23 Mean |Gradient|: 0.0272018909
Layer 24 Mean |Gradient|: 0.0534393750
Layer 25 Mean |Gradient|: 0.7563199997
Layer 26 Mean |Gradient|:

## Part 2: Perceptron

The Perceptron (Rosenblatt, 1958) - binary classifier with step activation

In [ ]:
# Perceptron for AND gate
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 0, 0, 1])

# Simple perceptron
w = np.array([0.5, 0.5])
b = -0.7

# Predictions
z = X @ w + b
#print("Z is ", z)
predictions = (z >= 0).astype(int)

print('AND Gate Results:')
for xi, yi, pred in zip(X, y, predictions):
    print(f'Input: {xi}, True: {yi}, Pred: {pred}')

Z is  [-0.7 -0.2 -0.2  0.3]
AND Gate Results:
Input: [0 0], True: 0, Pred: 0
Input: [0 1], True: 0, Pred: 0
Input: [1 0], True: 0, Pred: 0
Input: [1 1], True: 1, Pred: 1


## Summary

This notebook demonstrates:
- Artificial neuron computation
- Perceptron implementation
- Linear algebra representation

**Next:** Complete B05 for full neural network implementation

**Resources:**
- [COMPSCI 714 Complete Guide](../documentation/courses/COMPSCI_714_COMPLETE_GUIDE.md)